In [ ]:
from glider import IGCFile
from glider import GoProVideoMetadata

igc = IGCFile("2026-05-01-XTR-A68A17562166-01.IGC")
meta0 = GoProVideoMetadata.from_video("GX020107.MP4")
meta1 = GoProVideoMetadata.from_video("GX020107.MP4")

In [ ]:

## Python Implementation

import numpy as np
class AltitudeFusionKF:
    def __init__(self, dt=1):
        # State: [Altitude, Pressure_Bias]
        self.x = np.array([[0.0], [0.0]]) 
        self.P = np.eye(2) * 10.0
        
        # State Transition (Altitude = Altitude_old + Bias_old * 0)
        self.F = np.array([[1.0, 0.0],
                           [0.0, 1.0]])
        
        # Process Noise (Trust altitude to move, bias to stay stable)
        self.Q = np.array([[0.0001, 0.0], 
                           [0.0, 0.1]]) 
        
        # Measurement Matrices
        self.H_gps = np.array([[1.0, 1.0]])    # GPS measures altitude
        self.H_press = np.array([[0.0, 1.0]])  # Press measures altitude + bias
        
        # Measurement Noise
        self.R_gps = np.array([[0.5]])   # GPS is noisy
        self.R_press = np.array([[5.0]]) # Pressure is precise (relatively)

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q

    def update_gps(self, z_gps):
        z = np.array([[z_gps]])
        y = z - (self.H_gps @ self.x)
        S = self.H_gps @ self.P @ self.H_gps.T + self.R_gps
        K = self.P @ self.H_gps.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(2) - K @ self.H_gps) @ self.P

    def update_press(self, z_press):
        z = np.array([[z_press]])
        y = z - (self.H_press @ self.x)
        S = self.H_press @ self.P @ self.H_press.T + self.R_press
        K = self.P @ self.H_press.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(2) - K @ self.H_press) @ self.P

    def get_altitude(self):
        return self.x[0, 0]

## How to use it

#    1. Call predict() at every time step (e.g., 10Hz or 50Hz).
#    2. Call update_press() whenever you get a pressure reading (fast).
#    3. Call update_gps() only when the GPS provides a new coordinate (slow).

# This structure ensures that even when the pressure sensor starts to drift by 10 meters, the GPS update will pull the Pressure_Bias state in the opposite direction, effectively "zeroing out" the sensor error in real-time.
# Should I explain how to add vertical velocity (climb rate) to this state for even smoother tracking?

import numpy as np
def simple_altitude_kalman(gps_alt, pressure_alt, dt=1.0):
    # State: [Altitude]
    # Prediction: x = x (constant altitude model for simplicity)
    # Process noise (Q): How much we trust the model/how much altitude can change
    # Measurement noise (R): GPS (slow, stable) vs Pressure (fast, drift-prone)
    
    x = pressure_alt[0] # Initial state
    P = 1.0 # Initial uncertainty
    
    Q = 0.01 # Process noise
    R_gps = 0.5 # GPS measurement noise (less trust in individual samples)
    R_press = 5 # Pressure measurement noise (high trust in relative change)
    
    estimates = []
    
    for g, p in zip(gps_alt, pressure_alt):
        # 1. Predict
        x = x 
        P = P + Q
        
        # 2. Update with Pressure (Fast, frequent)
        K_p = P / (P + R_press)
        x = x + K_p * (p - x)
        P = (1 - K_p) * P
        
        # 3. Update with GPS (Corrects the drift)
        K_g = P / (P + R_gps)
        x = x + K_g * (g - x)
        P = (1 - K_g) * P
        
        estimates.append(x)
    return estimates

# Example 
# 
gps_data = [100, 101, 102, 103, 104]
press_data = [102, 105, 108, 110, 112] # Drifting away
# 
filtered = simple_altitude_kalman(igc.altitude_df['alt_gps'], igc.altitude_df['alt_pressure'])
print(f"Filtered: {filtered}")

from matplotlib import pyplot as plt
plt.plot(igc.altitude_df['alt_gps'])
plt.plot(igc.altitude_df['alt_pressure'])
plt.plot(filtered)
max(filtered)

In [ ]:
from PIL import Image
import cv2
import numpy as np

compass_position = (10,10)
compass_background = Image.open("compass_background.png").convert("RGBA").resize((200, 200), Image.Resampling.LANCZOS)
needle = Image.open("compass_needle.png").convert("RGBA").resize((200, 200), Image.Resampling.LANCZOS)
needle_center = [x/2 for x in needle.size]

def draw_compass(frame: np.array, heading: float):
    img = Image.fromarray(frame)
    rotated_overlay = needle.rotate(-heading, center=needle_center, resample=Image.BICUBIC)
    img.paste(compass_background, compass_position, compass_background)
    img.paste(rotated_overlay, compass_position, rotated_overlay)
    frame = np.array(img)
    return frame



In [ ]:
import datetime
from zoneinfo import ZoneInfo
nowutc = datetime.datetime.now(tz=datetime.UTC)

In [ ]:
import re
import os
pattern = r"G[XH](\d{2})(\d{4})"
for f in os.listdir("."):
    match = re.search(pattern, f)
    if match:
        chapter_zz = match.group(1)
        file_id_xxxx = match.group(2)
        print(f"File: {f} -> Chapter: {chapter_zz}, ID: {file_id_xxxx}")

In [ ]:
import datetime
backwards, forwards = igc.get_altitude_progression(time)

In [ ]:
def draw_altitude_line(igc: IGCFile, time: datetime, frame):
    backwards, forwards = igc.get_altitude_progression(time)
    y = [*-backwards, *-forwards]
    y = y + min(y)
    x = [x for x in range(len(y))]
    x_image_size = 280
    y_image_size = 100
    plot_position = (240, 90)
    x_scaled = plot_position[0] + (x - np.min(x)) / (np.max(x) - np.min(x)) * x_image_size
    y_scaled = plot_position[1] + (y - np.min(y)) / (np.max(y) - np.min(y)) * y_image_size

    x = x_scaled
    y = y_scaled

    points = np.column_stack((x, y)).astype(np.int32)

    # 3. Draw the Fading Shadow (Negative Y direction)
    # In image coordinates, "down" is the positive Y-axis.
    shadow_layers = 25
    max_alpha = 0.2

    for i in range(shadow_layers, 0, -1):
        # Create a transparent overlay for the current shadow layer
        overlay = frame.copy()
        
        # Calculate transparency: fades out as it moves away from the line
        alpha = max_alpha * (1 - i / shadow_layers)**1.5
        
        # Offset the points downward to create the shadow effect
        offset_points = points.copy()
        offset_points[:, 1] += i
        
        # Draw the shadow line on the overlay
        cv2.polylines(overlay, [offset_points], isClosed=False, color=(75, 150, 200), thickness=5)
        
        # Blend the overlay back into the main image
        frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

    # 4. Draw the main curve on top
    cv2.polylines(frame, [points[0:int(len(backwards))]], isClosed=False, color=(0, 0, 0), thickness=4, lineType=cv2.LINE_AA)
    cv2.polylines(frame, [points[int(len(backwards)):]], isClosed=False, color=(0, 255, 0), thickness=4, lineType=cv2.LINE_AA)
    return frame

In [ ]:
sample_time = datetime.datetime(year=2026, month=5, day=1, hour=10, minute=2, second=0, tzinfo=datetime.UTC)
backwards, forwards = igc.get_altitude_progression(sample_time)
alt = igc.get_gps_altitude_at_time(sample_time)
forwards[0],alt

In [ ]:
[*-backwards,*-forwards]

In [ ]:


timestring = sample_time.replace(microsecond=0).astimezone(ZoneInfo("Europe/Zurich")).isoformat()
img = Image.open("frame.jpg")
frame = np.array(img)
frame = cv2.putText(
    frame,
    timestring,
    (25, frame.shape[0]-25), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 3)
frame = cv2.putText(
    frame,
    "ALT 1234 m",
    (240, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
frame = draw_compass(frame, 90)
frame = draw_altitude_line(igc, sample_time, frame)

Image.fromarray(frame)

In [ ]:
import pandas as pd
import numpy as np

bearing_df = pd.DataFrame([
    {"ts": x.gpx_track_point.time.timestamp(), 
     "b_x": np.cos(x.bearing.bearing_rad),
     "b_y": np.sin(x.bearing.bearing_rad)} for x in igc.records
])

In [ ]:
igc.records[4].bearing

In [ ]:
import datetime
igc.get_altitude_at_time(datetime.datetime.fromtimestamp(1777629.0))

In [ ]:
igc.bearing_df['ts'].iloc[760]
